# RHINO Diagnostic Notebook
## ADC Non-linearity, Harmonic Analysis, and Raw Time Stream
### Run cells top to bottom. Each cell is independent.

In [1]:
# ================================================================
# CELL 1 — Imports and paths
# ================================================================
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os

BASE  = '/Users/user/Downloads/Manny-Masters/Project/Data'
OUT   = BASE + '/rhino_figures_60_85MHz'
os.makedirs(OUT, exist_ok=True)

# Dataset paths
P2  = BASE + '/Jodrell_Discone/LNA'           # DS2: discone + LNA, 17 Jun
P4  = BASE + '/Jodrell_load/LNA'              # DS4: load + LNA, 19 Jun
P5  = BASE + '/Jodrell_Discone_New/LNA'       # DS5: discone + LNA, 22 Jun
P6  = BASE + '/Jodrell_Discone_New/No_LNA'    # DS6: discone no LNA, 22 Jun

FS_MHZ      = 4423.680
N_HIRES     = 1048576
ADC_BITS    = 14
ADC_MAX     = 2**(ADC_BITS - 1) - 1   # 8191 for 14-bit
CLIP_THRESH = 0.95                     # flag if |sample| > 95% of ADC_MAX

plt.rcParams.update({
    'figure.dpi': 150, 'figure.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.4, 'font.size': 10
})

def save_fig(fig, name):
    p = os.path.join(OUT, name)
    fig.savefig(p, dpi=150, bbox_inches='tight')
    print('  Saved:', p)
    plt.close(fig)

print('PASS imports OK')
print('ADC_MAX (14-bit):', ADC_MAX)

PASS imports OK
ADC_MAX (14-bit): 8191


In [2]:
# ================================================================
# CELL 2 — ADC clipping check
# Checks raw_hires for each dataset.
# Clipping (|sample| >= ADC_MAX) causes harmonic distortion.
# Even near-clipping (> 95% of ADC_MAX) can introduce non-linearity.
# ================================================================
print('ADC CLIPPING DIAGNOSTIC')
print('=' * 55)

raw_files = [
    (P2,  'raw_hires_20250617_232907.npy', 'DS2: Discone + LNA (17 Jun)'),
    (P4,  'raw_hires_20250619_211156.npy', 'DS4: Load + LNA (19 Jun)'),
    (P5,  'raw_hires_20250622_030852.npy', 'DS5: Discone + LNA (22 Jun)'),
    (P6,  'raw_hires_20250622_040048.npy', 'DS6: Discone no LNA (22 Jun)'),
]

for path, fname, label in raw_files:
    full = os.path.join(path, fname)
    if not os.path.exists(full):
        print('  SKIP %-40s not found' % label)
        continue
    raw = np.load(full, allow_pickle=False)
    raw = raw.astype(np.float32)
    n_samples   = len(raw)
    peak_abs    = float(np.max(np.abs(raw)))
    n_clipped   = int(np.sum(np.abs(raw) >= ADC_MAX))
    n_near_clip = int(np.sum(np.abs(raw) >= CLIP_THRESH * ADC_MAX))
    rms         = float(np.sqrt(np.mean(raw**2)))
    clip_pct    = 100.0 * n_clipped / n_samples
    near_pct    = 100.0 * n_near_clip / n_samples
    status      = 'CLIPPING' if n_clipped > 0 else ('WARN near-clip' if n_near_clip > 10 else 'OK')
    print()
    print('  %s' % label)
    print('    Samples       : %d' % n_samples)
    print('    Peak |sample| : %.1f  (ADC_MAX = %d)' % (peak_abs, ADC_MAX))
    print('    RMS           : %.2f ADU'  % rms)
    print('    Clipped       : %d  (%.4f%%)' % (n_clipped, clip_pct))
    print('    Near-clip     : %d  (%.4f%%)' % (n_near_clip, near_pct))
    print('    STATUS        : %s' % status)

print()
print('PASS clipping check complete')

ADC CLIPPING DIAGNOSTIC

  DS2: Discone + LNA (17 Jun)
    Samples       : 1048576
    Peak |sample| : 5183.0  (ADC_MAX = 8191)
    RMS           : 309.02 ADU
    Clipped       : 0  (0.0000%)
    Near-clip     : 0  (0.0000%)
    STATUS        : OK

  DS4: Load + LNA (19 Jun)
    Samples       : 1048576
    Peak |sample| : 14.0  (ADC_MAX = 8191)
    RMS           : 4.42 ADU
    Clipped       : 0  (0.0000%)
    Near-clip     : 0  (0.0000%)
    STATUS        : OK

  DS5: Discone + LNA (22 Jun)
    Samples       : 1048576
    Peak |sample| : 2729.0  (ADC_MAX = 8191)
    RMS           : 215.16 ADU
    Clipped       : 0  (0.0000%)
    Near-clip     : 0  (0.0000%)
    STATUS        : OK

  DS6: Discone no LNA (22 Jun)
    Samples       : 1048576
    Peak |sample| : 422.0  (ADC_MAX = 8191)
    RMS           : 27.77 ADU
    Clipped       : 0  (0.0000%)
    Near-clip     : 0  (0.0000%)
    STATUS        : OK

PASS clipping check complete


In [3]:
# ================================================================
# CELL 3 — Harmonic analysis
# The spike near 82 MHz: is it a harmonic of a lower frequency?
# We check whether the DS4 (load+LNA, no antenna) spectrum also
# shows a spike at 82 MHz. If yes -> instrument-generated (ADC
# non-linearity or board self-interference).
# If no -> environmental (sky or antenna-coupled signal).
# We also compute the implied fundamental frequencies.
# ================================================================
print('HARMONIC / SPURIOUS SPIKE ANALYSIS')
print('=' * 55)

SPIKE_MHZ   = 82.0   # approximate spike frequency from Fig 7
SEARCH_WIN  = 2.0    # MHz window to search for spike peak

# Harmonic candidates: if 82 MHz is the Nth harmonic,
# the fundamental is at 82/N MHz
print('  If spike at %.1f MHz is a harmonic:' % SPIKE_MHZ)
for N in range(2, 8):
    fund = SPIKE_MHZ / N
    print('    N=%d  ->  fundamental at %.3f MHz' % (N, fund))

print()

# Load hi-res spectra and check spike amplitude in each dataset
hires_files = [
    (P2,  'freq_hires_20250617_232907.npy', 'fft_hires_20250617_232907.npy',
     'DS2: Discone + LNA (17 Jun)'),
    (P4,  'freq_hires_20250619_211156.npy', 'fft_hires_20250619_211156.npy',
     'DS4: Load + LNA (19 Jun) — KEY CALIBRATION'),
    (P5,  'freq_hires_20250622_030852.npy', 'fft_hires_20250622_030852.npy',
     'DS5: Discone + LNA (22 Jun)'),
    (P6,  'freq_hires_20250622_040048.npy', 'fft_hires_20250622_040048.npy',
     'DS6: Discone no LNA (22 Jun)'),
]

spike_results = {}
for path, ffreq, fspec, label in hires_files:
    ff = os.path.join(path, ffreq)
    fs = os.path.join(path, fspec)
    if not os.path.exists(ff) or not os.path.exists(fs):
        print('  SKIP %s' % label)
        continue
    freq = np.load(ff, allow_pickle=False)
    spec = np.load(fs, allow_pickle=False)
    # Noise floor: median in 60-78 MHz (below the spike, avoids FM)
    floor_mask  = (freq >= 60.0) & (freq <= 78.0)
    noise_floor = float(np.median(spec[floor_mask]))
    # Search for spike peak near SPIKE_MHZ
    spike_mask  = (freq >= SPIKE_MHZ - SEARCH_WIN) & \
                  (freq <= SPIKE_MHZ + SEARCH_WIN)
    if spike_mask.sum() == 0:
        print('  SKIP %s — no bins in search window' % label)
        continue
    spike_peak_db   = float(np.max(spec[spike_mask]))
    spike_peak_freq = float(freq[spike_mask][np.argmax(spec[spike_mask])])
    spike_snr       = spike_peak_db - noise_floor
    spike_results[label] = {
        'peak_freq': spike_peak_freq,
        'peak_db'  : spike_peak_db,
        'floor_db' : noise_floor,
        'snr_db'   : spike_snr,
    }
    print('  %s' % label)
    print('    Noise floor (60-78 MHz median) : %.2f dB' % noise_floor)
    print('    Spike peak frequency           : %.4f MHz' % spike_peak_freq)
    print('    Spike peak power               : %.2f dB' % spike_peak_db)
    print('    Spike SNR above floor          : %.2f dB' % spike_snr)
    if spike_snr > 5.0:
        print('    VERDICT: SPIKE DETECTED (>5 dB above floor)')
    else:
        print('    VERDICT: no significant spike at this frequency')
    print()

# Key interpretation
print('KEY INTERPRETATION:')
ds4_key = [k for k in spike_results if 'DS4' in k]
if ds4_key:
    ds4_snr = spike_results[ds4_key[0]]['snr_db']
    if ds4_snr > 5.0:
        print('  Spike ALSO present in DS4 (load+LNA, no antenna)')
        print('  -> Likely INSTRUMENT-GENERATED (ADC non-linearity')
        print('     or board self-interference), NOT a sky/antenna signal.')
    else:
        print('  Spike NOT present in DS4 (load+LNA, no antenna)')
        print('  -> Likely ENVIRONMENTAL (real sky or antenna-coupled')
        print('     RFI), not caused by ADC non-linearity.')
        print('  -> Harmonic hypothesis requires a real transmitter')
        print('     at the implied fundamental frequency.')
print()
print('PASS harmonic analysis complete')

HARMONIC / SPURIOUS SPIKE ANALYSIS
  If spike at 82.0 MHz is a harmonic:
    N=2  ->  fundamental at 41.000 MHz
    N=3  ->  fundamental at 27.333 MHz
    N=4  ->  fundamental at 20.500 MHz
    N=5  ->  fundamental at 16.400 MHz
    N=6  ->  fundamental at 13.667 MHz
    N=7  ->  fundamental at 11.714 MHz

  DS2: Discone + LNA (17 Jun)
    Noise floor (60-78 MHz median) : 58.43 dB
    Spike peak frequency           : 82.1138 MHz
    Spike peak power               : 63.14 dB
    Spike SNR above floor          : 4.70 dB
    VERDICT: no significant spike at this frequency

  DS4: Load + LNA (19 Jun) — KEY CALIBRATION
    Noise floor (60-78 MHz median) : 7.88 dB
    Spike peak frequency           : 82.1180 MHz
    Spike peak power               : 10.82 dB
    Spike SNR above floor          : 2.94 dB
    VERDICT: no significant spike at this frequency

  DS5: Discone + LNA (22 Jun)
    Noise floor (60-78 MHz median) : 44.32 dB
    Spike peak frequency           : 82.1138 MHz
    Spike peak 

In [4]:
# ================================================================
# CELL 4 — Raw time stream plot
# Plots the raw ADC voltage samples x[n] vs time for DS5 and DS6.
# This is what Phil asked for. Shows the actual ADC output before
# any spectral processing.
# ================================================================
print('RAW TIME STREAM PLOT')
print('=' * 55)

# How many samples to plot (too many = unreadable)
# 4096 samples = ~0.93 microseconds at 4423.68 Msps
N_PLOT      = 4096
# Time axis in microseconds
dt_us       = 1.0 / FS_MHZ   # microseconds per sample

raw_stream_files = [
    (P5, 'raw_hires_20250622_030852.npy', 'DS5: Discone + ZKL-2+ LNA (22 Jun)'),
    (P6, 'raw_hires_20250622_040048.npy', 'DS6: Discone, No LNA (22 Jun)'),
]

fig, axes = plt.subplots(len(raw_stream_files), 1,
                          figsize=(12, 4 * len(raw_stream_files)))
if len(raw_stream_files) == 1:
    axes = [axes]

for ax, (path, fname, label) in zip(axes, raw_stream_files):
    full = os.path.join(path, fname)
    if not os.path.exists(full):
        ax.text(0.5, 0.5, 'File not found:\n%s' % fname,
                ha='center', va='center', transform=ax.transAxes)
        ax.set_title(label)
        continue
    raw  = np.load(full, allow_pickle=False)
    seg  = raw[:N_PLOT].astype(np.float32)
    t_us = np.arange(N_PLOT) * dt_us
    rms  = float(np.sqrt(np.mean(raw.astype(np.float32)**2)))
    peak = float(np.max(np.abs(raw.astype(np.float32))))
    ax.plot(t_us, seg, lw=0.4, color='steelblue', alpha=0.8)
    ax.axhline( ADC_MAX, color='red', ls='--', lw=1.0, alpha=0.7,
               label='ADC clip level (+%d ADU)' % ADC_MAX)
    ax.axhline(-ADC_MAX, color='red', ls='--', lw=1.0, alpha=0.7)
    ax.axhline(0, color='grey', ls='-', lw=0.5, alpha=0.4)
    ax.set_xlabel('Time (µs)')
    ax.set_ylabel('ADC output (ADU)')
    ax.set_title('%s\nRMS = %.1f ADU  |  Peak = %.0f ADU  |  '
                 'ADC_MAX = %d ADU  |  First %d samples shown'
                 % (label, rms, peak, ADC_MAX, N_PLOT))
    ax.set_ylim(-ADC_MAX * 1.1, ADC_MAX * 1.1)
    ax.legend(fontsize=8, loc='upper right')
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    print('  %s' % label)
    print('    Total samples : %d' % len(raw))
    print('    RMS           : %.2f ADU' % rms)
    print('    Peak          : %.0f ADU  (ADC_MAX = %d)' % (peak, ADC_MAX))
    print('    Peak / ADC_MAX: %.3f  (%s)' % (
          peak / ADC_MAX,
          'CLIPPING' if peak >= ADC_MAX else
          'near-clip' if peak >= 0.95 * ADC_MAX else 'OK'))
    print()

fig.suptitle('Raw ADC Time Stream — RHINO Science Band Observations\n'
             'First %d samples shown  |  '
             'Sample rate = %.3f Msps  |  '
             'dt = %.4f µs/sample' % (N_PLOT, FS_MHZ, dt_us),
             fontsize=10)
fig.tight_layout()
save_fig(fig, 'diag_raw_timestream.png')
print('PASS raw time stream plot saved')

RAW TIME STREAM PLOT
  DS5: Discone + ZKL-2+ LNA (22 Jun)
    Total samples : 1048576
    RMS           : 215.16 ADU
    Peak          : 2729 ADU  (ADC_MAX = 8191)
    Peak / ADC_MAX: 0.333  (OK)

  DS6: Discone, No LNA (22 Jun)
    Total samples : 1048576
    RMS           : 27.77 ADU
    Peak          : 422 ADU  (ADC_MAX = 8191)
    Peak / ADC_MAX: 0.052  (OK)

  Saved: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_figures_60_85MHz/diag_raw_timestream.png
PASS raw time stream plot saved


In [5]:
# ================================================================
# CELL 5 — Wideband power spectrum of raw time stream
# Computes the full 0-2212 MHz FFT of the raw samples.
# This shows where the power actually sits in the band and
# whether there are strong out-of-band signals that could
# alias or cause non-linearity.
# ================================================================
print('WIDEBAND POWER SPECTRUM OF RAW TIME STREAM')
print('=' * 55)

wideband_files = [
    (P5, 'raw_hires_20250622_030852.npy', 'DS5: Discone + LNA (22 Jun)'),
    (P6, 'raw_hires_20250622_040048.npy', 'DS6: Discone no LNA (22 Jun)'),
]

freq_axis = np.fft.rfftfreq(N_HIRES, d=1.0/FS_MHZ)   # MHz

fig, axes = plt.subplots(len(wideband_files), 1,
                          figsize=(12, 4 * len(wideband_files)))
if len(wideband_files) == 1:
    axes = [axes]

for ax, (path, fname, label) in zip(axes, wideband_files):
    full = os.path.join(path, fname)
    if not os.path.exists(full):
        ax.text(0.5, 0.5, 'File not found', ha='center', va='center',
                transform=ax.transAxes)
        continue
    raw  = np.load(full, allow_pickle=False).astype(np.float32)
    # Hann window + FFT
    win  = np.hanning(N_HIRES).astype(np.float32)
    win /= np.sqrt(np.mean(win**2))
    spec = np.abs(np.fft.rfft(raw * win))**2
    spec_db = 10 * np.log10(np.maximum(spec, 1e-10))
    # Plot 0-500 MHz
    wb_mask = freq_axis <= 500.0
    ax.plot(freq_axis[wb_mask], spec_db[wb_mask],
            lw=0.4, color='steelblue', alpha=0.85)
    # Mark RHINO band and FM band
    ax.axvspan(60, 85,    alpha=0.12, color='gold',  label='RHINO (60-85 MHz)')
    ax.axvspan(87.5, 108, alpha=0.08, color='red',   label='FM (87.5-108 MHz)')
    ax.axvline(82.0, color='orange', ls='--', lw=1.2, alpha=0.8,
               label='Spike at 82 MHz')
    # Mark harmonic candidates
    for N_harm in [2, 3, 4]:
        fund = 82.0 / N_harm
        ax.axvline(fund, color='purple', ls=':', lw=0.8, alpha=0.6)
        ax.text(fund, ax.get_ylim()[0] if ax.get_ylim()[0] != 0 else spec_db[wb_mask].min(),
                '82/%d\n=%.1f' % (N_harm, fund),
                fontsize=7, color='purple', ha='center', va='bottom')
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel('Power (dB, arb.)')
    ax.set_title('%s — Wideband power spectrum from raw ADC samples\n'
                 'Purple dotted lines = harmonic fundamental candidates for 82 MHz spike'
                 % label)
    ax.legend(fontsize=8, loc='upper right')
    ax.set_xlim(0, 500)
    # Print peak frequencies above threshold
    threshold = float(np.median(spec_db[wb_mask])) + 20.0
    peak_mask = spec_db[wb_mask] > threshold
    peak_freqs = freq_axis[wb_mask][peak_mask]
    print('  %s' % label)
    print('    Peaks > median+20dB in 0-500 MHz:')
    if len(peak_freqs) > 0:
        # Cluster peaks
        prev = -999
        for pf in peak_freqs:
            if pf - prev > 0.5:   # new cluster
                print('      %.3f MHz' % pf)
            prev = pf
    else:
        print('      None found above threshold')
    print()

fig.suptitle('Wideband Power Spectrum from Raw ADC Samples (0-500 MHz)\n'
             'Used to identify harmonic fundamentals and out-of-band interference',
             fontsize=10)
fig.tight_layout()
save_fig(fig, 'diag_wideband_raw_spectrum.png')
print('PASS wideband raw spectrum saved')

WIDEBAND POWER SPECTRUM OF RAW TIME STREAM
  DS5: Discone + LNA (22 Jun)
    Peaks > median+20dB in 0-500 MHz:
      0.000 MHz
      3.873 MHz
      6.261 MHz
      92.602 MHz
      93.146 MHz
      93.694 MHz
      94.791 MHz
      96.002 MHz
      96.985 MHz
      97.803 MHz
      98.356 MHz
      99.723 MHz
      101.094 MHz
      102.866 MHz
      104.106 MHz
      105.751 MHz
      106.300 MHz
      106.848 MHz
      107.355 MHz
      108.203 MHz
      110.135 MHz
      111.506 MHz
      112.324 MHz
      113.421 MHz
      114.792 MHz
      115.615 MHz
      116.712 MHz
      118.079 MHz
      119.998 MHz
      121.998 MHz
      139.396 MHz
      140.560 MHz
      141.919 MHz
      189.084 MHz
      192.000 MHz
      232.002 MHz
      233.453 MHz
      234.546 MHz
      235.634 MHz
      240.000 MHz
      280.003 MHz
      296.000 MHz
      304.003 MHz
      335.509 MHz
      337.534 MHz
      440.003 MHz
      496.007 MHz

  DS6: Discone no LNA (22 Jun)
    Peaks > median+20dB in

In [6]:
# ================================================================
# CELL 6 — Pure Gaussian noise normalisation test
# Feeds identical Gaussian random samples to both FFT and PFB.
# On a flat-spectrum input the two spectrometers should agree
# to within the theoretical ENBW offset of +1.76 dB.
# This confirms the normalisation is correct.
# ================================================================
print('PURE GAUSSIAN NOISE NORMALISATION TEST')
print('=' * 55)

N_FFT   = 16384    # coarse
N_TAPS  = 4
N_AVG   = 200      # number of spectra to average for stable estimate
rng     = np.random.default_rng(42)
sigma   = 100.0

fft_spectra = []
pfb_spectra = []

# Prototype PFB filter
M_p     = N_TAPS * N_FFT
t_p     = np.arange(M_p) - M_p // 2
proto   = (np.sinc(t_p / N_FFT) * np.hanning(M_p)).astype(np.float32)
proto  /= np.sqrt(np.mean(proto**2))

# Hann window
win     = np.hanning(N_FFT).astype(np.float32)
win    /= np.sqrt(np.mean(win**2))

for _ in range(N_AVG):
    noise = rng.normal(0, sigma, N_FFT + M_p).astype(np.float32)
    # FFT
    fft_out = np.abs(np.fft.rfft(noise[-N_FFT:] * win))**2
    fft_spectra.append(fft_out)
    # PFB
    blocks  = noise[:M_p].reshape(N_TAPS, N_FFT)
    pfb_f   = np.zeros(N_FFT, dtype=np.float32)
    for t in range(N_TAPS):
        pfb_f += blocks[t] * proto[t*N_FFT:(t+1)*N_FFT]
    pfb_out = np.abs(np.fft.rfft(pfb_f))**2
    pfb_spectra.append(pfb_out)

fft_mean = 10 * np.log10(np.mean(fft_spectra, axis=0))
pfb_mean = 10 * np.log10(np.mean(pfb_spectra, axis=0))

# Exclude DC (bin 0) and Nyquist (last bin)
fft_level = float(np.mean(fft_mean[1:-1]))
pfb_level = float(np.mean(pfb_mean[1:-1]))
measured_offset = pfb_level - fft_level
theoretical_offset = 10 * np.log10(1.5 / 1.0)   # ENBW_Hann / ENBW_PFB

print('  N_AVG spectra       : %d' % N_AVG)
print('  FFT mean level      : %.4f dB' % fft_level)
print('  PFB mean level      : %.4f dB' % pfb_level)
print('  Measured PFB-FFT    : %+.4f dB' % measured_offset)
print('  Theoretical (ENBW)  : %+.4f dB' % theoretical_offset)
print('  Residual            : %+.4f dB' % (measured_offset - theoretical_offset))
if abs(measured_offset - theoretical_offset) < 0.1:
    print('  VERDICT: normalisation CORRECT — offset matches ENBW prediction')
else:
    print('  VERDICT: WARN — offset does not match ENBW prediction')
    print('           Check prototype filter normalisation.')

# Plot
freq_c = np.fft.rfftfreq(N_FFT, d=1.0/FS_MHZ)
fig, axes = plt.subplots(2, 1, figsize=(10, 7))
axes[0].plot(freq_c[1:-1], fft_mean[1:-1], color='steelblue',
             lw=0.6, alpha=0.8, label='FFT (Hann)  mean=%.2f dB' % fft_level)
axes[0].plot(freq_c[1:-1], pfb_mean[1:-1], color='firebrick',
             lw=0.6, alpha=0.8, label='PFB (4 taps)  mean=%.2f dB' % pfb_level)
axes[0].axhline(fft_level, color='steelblue', ls=':', lw=1.0, alpha=0.6)
axes[0].axhline(pfb_level, color='firebrick', ls=':', lw=1.0, alpha=0.6)
axes[0].set_xlabel('Frequency (MHz)')
axes[0].set_ylabel('Power (dB, arb.)')
axes[0].set_title('Wideband (0-2212 MHz) — N=%d averaged Gaussian noise spectra' % N_AVG)
axes[0].legend(fontsize=9)

# Zoom to RHINO band
rhino_mask = (freq_c >= 60) & (freq_c <= 85)
axes[1].plot(freq_c[rhino_mask], fft_mean[rhino_mask], color='steelblue',
             lw=1.0, label='FFT (Hann)')
axes[1].plot(freq_c[rhino_mask], pfb_mean[rhino_mask], color='firebrick',
             lw=1.0, label='PFB (4 taps)')
axes[1].axhline(fft_level, color='steelblue', ls=':', lw=1.0, alpha=0.6)
axes[1].axhline(pfb_level, color='firebrick', ls=':', lw=1.0, alpha=0.6)
axes[1].set_xlabel('Frequency (MHz)')
axes[1].set_ylabel('Power (dB, arb.)')
axes[1].set_title('RHINO band zoom (60-85 MHz)')
axes[1].legend(fontsize=9)

fig.suptitle(
    'Pure Gaussian Noise Normalisation Test\n'
    'Measured PFB-FFT offset = %+.4f dB  |  '
    'Theoretical (ENBW) = %+.4f dB  |  '
    'Residual = %+.4f dB'
    % (measured_offset, theoretical_offset,
       measured_offset - theoretical_offset),
    fontsize=10)
fig.tight_layout()
save_fig(fig, 'diag_gaussian_normalisation_test.png')
print('PASS normalisation test saved')

PURE GAUSSIAN NOISE NORMALISATION TEST
  N_AVG spectra       : 200
  FFT mean level      : 82.1336 dB
  PFB mean level      : 88.1497 dB
  Measured PFB-FFT    : +6.0161 dB
  Theoretical (ENBW)  : +1.7609 dB
  Residual            : +4.2551 dB
  VERDICT: WARN — offset does not match ENBW prediction
           Check prototype filter normalisation.
  Saved: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_figures_60_85MHz/diag_gaussian_normalisation_test.png
PASS normalisation test saved


In [7]:
# ================================================================
# CELL 7 — 4 taps vs 8 taps PFB filter response comparison
# Answers Phil's question: does increasing taps help?
# ================================================================
print('PFB TAPS COMPARISON: 4 vs 8')
print('=' * 55)

N_FFT   = 16384
nfft_pad = 2**20   # zero-padding for frequency resolution

# Hann FFT channel response
win_hann = np.hanning(N_FFT)
win_hann /= win_hann.sum()
W_fft = np.abs(np.fft.fft(win_hann, nfft_pad))
W_fft_db = 20 * np.log10(np.maximum(W_fft / W_fft.max(), 1e-12))

tap_configs = []
for K in [4, 8]:
    M   = K * N_FFT
    t   = np.arange(M) - M // 2
    h   = np.sinc(t / N_FFT) * np.hanning(M)
    h  /= h.sum()
    W   = np.abs(np.fft.fft(h, nfft_pad * K)[:nfft_pad])
    W_db = 20 * np.log10(np.maximum(W / W.max(), 1e-12))
    # Find first sidelobe level (search between 1.0 and 3.0 bins offset)
    bins_axis = np.fft.fftfreq(nfft_pad) * N_FFT
    bins_axis = np.abs(bins_axis)
    sl_mask = (bins_axis >= 1.0) & (bins_axis <= 3.5)
    first_sl = float(W_db[sl_mask].max())
    tap_configs.append({
        'K': K, 'W_db': W_db, 'first_sl': first_sl
    })
    print('  K=%d taps: first sidelobe = %.1f dB' % (K, first_sl))

# FFT sidelobe
bins_fft = np.abs(np.fft.fftfreq(nfft_pad) * N_FFT)
sl_mask_fft = (bins_fft >= 0.5) & (bins_fft <= 3.5)
fft_sl = float(W_fft_db[sl_mask_fft].max())
print('  FFT (Hann): channel boundary response = %.1f dB' % fft_sl)

# Plot
xlim  = 4.0
bins_plot = np.fft.fftshift(np.fft.fftfreq(nfft_pad) * N_FFT)
W_fft_shift = np.fft.fftshift(W_fft_db)
sel   = np.abs(bins_plot) <= xlim

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(bins_plot[sel], W_fft_shift[sel],
        color='steelblue', lw=1.5, label='FFT (Hann)  boundary=%.1f dB' % fft_sl)
colours = ['firebrick', 'darkorange']
for cfg, col in zip(tap_configs, colours):
    W_shift = np.fft.fftshift(cfg['W_db'])
    ax.plot(bins_plot[sel], W_shift[sel], color=col, lw=1.5,
            label='PFB K=%d taps  sidelobe=%.1f dB' % (cfg['K'], cfg['first_sl']))
ax.axvline( 0.5, color='grey', ls='--', lw=0.8, alpha=0.5)
ax.axvline(-0.5, color='grey', ls='--', lw=0.8, alpha=0.5)
ax.set_xlabel('Frequency offset (bins)')
ax.set_ylabel('Channel response (dB)')
ax.set_title('PFB Taps Comparison: FFT vs K=4 vs K=8\n'
             'Hann-windowed sinc prototype filter')
ax.set_xlim(-xlim, xlim)
ax.set_ylim(-100, 5)
ax.legend(fontsize=9)
fig.tight_layout()
save_fig(fig, 'diag_pfb_taps_comparison.png')
print('PASS taps comparison saved')
print()
print('INTERPRETATION:')
k4  = tap_configs[0]['first_sl']
k8  = tap_configs[1]['first_sl']
print('  K=4 -> K=8 sidelobe improvement: %.1f dB' % (k8 - k4))
print('  Trade-off: K=8 requires 2x more samples per spectrum')
print('  and 2x more computation. For the current RHINO science')
print('  case the improvement should be weighed against the')
print('  additional computational cost on the PYNQ CPU.')

PFB TAPS COMPARISON: 4 vs 8
  K=4 taps: first sidelobe = -1.0 dB
  K=8 taps: first sidelobe = 0.0 dB
  FFT (Hann): channel boundary response = -1.4 dB
  Saved: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_figures_60_85MHz/diag_pfb_taps_comparison.png
PASS taps comparison saved

INTERPRETATION:
  K=4 -> K=8 sidelobe improvement: 1.0 dB
  Trade-off: K=8 requires 2x more samples per spectrum
  and 2x more computation. For the current RHINO science
  case the improvement should be weighed against the
  additional computational cost on the PYNQ CPU.


In [8]:
# ================================================================
# CELL 8 — Hann vs Blackman window comparison
# Answers Phil's question: why Hann and not Blackman?
# ================================================================
print('WINDOW COMPARISON: Hann vs Blackman')
print('=' * 55)

N_FFT    = 16384
nfft_pad = 2**20

windows = {
    'Rectangular (boxcar)': np.ones(N_FFT),
    'Hann'                : np.hanning(N_FFT),
    'Hamming'             : np.hamming(N_FFT),
    'Blackman'            : np.blackman(N_FFT),
    'Blackman-Harris'     : np.blackman(N_FFT),  # approx
}

# Use scipy for Blackman-Harris
from scipy.signal import windows as sp_windows
windows['Blackman-Harris'] = sp_windows.blackmanharris(N_FFT)

print('  %-22s  %10s  %10s  %10s' %
      ('Window', 'ENBW(bins)', 'First SL(dB)', 'Main lobe(bins)'))
print('  ' + '-' * 60)

fig, ax = plt.subplots(figsize=(10, 5))
colours_w = ['grey', 'steelblue', 'green', 'firebrick', 'purple']
bins_axis = np.fft.fftshift(np.fft.fftfreq(nfft_pad) * N_FFT)
sel       = np.abs(bins_axis) <= 6.0

for (name, win), col in zip(windows.items(), colours_w):
    win_norm = win / win.sum()
    W        = np.abs(np.fft.fft(win_norm, nfft_pad))
    W_db     = 20 * np.log10(np.maximum(W / W.max(), 1e-12))
    W_shift  = np.fft.fftshift(W_db)
    # ENBW
    enbw     = N_FFT * np.sum(win**2) / np.sum(win)**2
    # First sidelobe (search 1-5 bins)
    bins_pos = np.fft.fftfreq(nfft_pad) * N_FFT
    sl_mask  = (bins_pos >= 1.0) & (bins_pos <= 5.0)
    first_sl = float(W_db[sl_mask].max())
    # Main lobe width (3dB)
    half_mask = (np.abs(bins_axis) <= 2.0)
    above3db  = bins_axis[half_mask & (W_shift >= -3.0)]
    ml_width  = float(above3db.max() - above3db.min()) if len(above3db) > 0 else 0
    print('  %-22s  %10.3f  %12.1f  %10.2f' %
          (name, enbw, first_sl, ml_width))
    ax.plot(bins_axis[sel], W_shift[sel], color=col, lw=1.3,
            label='%s (SL=%.0f dB, ENBW=%.2f)' % (name, first_sl, enbw))

ax.axvline( 0.5, color='grey', ls='--', lw=0.7, alpha=0.4)
ax.axvline(-0.5, color='grey', ls='--', lw=0.7, alpha=0.4)
ax.axhline(-13,  color='grey', ls=':',  lw=0.6, alpha=0.4)
ax.axhline(-58,  color='grey', ls=':',  lw=0.6, alpha=0.4)
ax.set_xlabel('Frequency offset (bins)')
ax.set_ylabel('Channel response (dB)')
ax.set_title('Window Function Comparison — Channel Frequency Response\n'
             'Trade-off between sidelobe suppression and main lobe width')
ax.set_xlim(-6, 6)
ax.set_ylim(-100, 5)
ax.legend(fontsize=8, loc='upper right')
fig.tight_layout()
save_fig(fig, 'diag_window_comparison.png')
print()
print('KEY RESULT FOR RHINO:')
print('  Hann: good sidelobe suppression with 2-bin main lobe.')
print('  Blackman: better sidelobe suppression but 3-bin main lobe.')
print('  At 270 kHz/bin coarse resolution, Blackman costs 1 extra')
print('  bin of spectral resolution (270 kHz) to gain ~27 dB more')
print('  sidelobe suppression over Hann. Whether this trade-off')
print('  is worthwhile depends on the RFI environment.')
print('  The PFB already achieves ~57 dB suppression, making the')
print('  window choice less critical for the PFB spectrometer.')
print('PASS window comparison saved')

WINDOW COMPARISON: Hann vs Blackman
  Window                  ENBW(bins)  First SL(dB)  Main lobe(bins)
  ------------------------------------------------------------
  Rectangular (boxcar)         1.000         -13.3        0.88
  Hann                         1.500          -6.0        1.44
  Hamming                      1.363          -7.4        1.28
  Blackman                     1.727          -4.5        1.62
  Blackman-Harris              2.004          -3.3        1.88
  Saved: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_figures_60_85MHz/diag_window_comparison.png

KEY RESULT FOR RHINO:
  Hann: good sidelobe suppression with 2-bin main lobe.
  Blackman: better sidelobe suppression but 3-bin main lobe.
  At 270 kHz/bin coarse resolution, Blackman costs 1 extra
  bin of spectral resolution (270 kHz) to gain ~27 dB more
  sidelobe suppression over Hann. Whether this trade-off
  is worthwhile depends on the RFI environment.
  The PFB already achieves ~57 dB suppression, 

In [9]:
import numpy as np

N_FFT_COARSE = 16384
N_TAPS       = 4
pfb_len      = N_FFT_COARSE * N_TAPS

t_pfb      = np.arange(pfb_len, dtype=np.float64) - pfb_len // 2
pfb_coeffs = np.sinc(t_pfb / N_FFT_COARSE) * np.hanning(pfb_len)
pfb_coeffs /= np.sum(pfb_coeffs.reshape(N_TAPS, N_FFT_COARSE), axis=0).mean()

fft_window      = np.hanning(N_FFT_COARSE).astype(np.float64)
fft_window_norm = float(np.sum(fft_window**2))

# Current (wrong)
pfb_win_norm_old = float(np.sum(pfb_coeffs**2))

# Fixed: norm of effective window
eff_window       = np.sum(pfb_coeffs.reshape(N_TAPS, N_FFT_COARSE), axis=0)
pfb_win_norm_new = float(np.sum(eff_window**2))

print('fft_window_norm      : %.4f' % fft_window_norm)
print('pfb_win_norm (old)   : %.4f' % pfb_win_norm_old)
print('pfb_win_norm (new)   : %.4f' % pfb_win_norm_new)
print()
print('Old norm offset      : %.4f dB' % (10*np.log10(pfb_win_norm_old/fft_window_norm)))
print('New norm offset      : %.4f dB' % (10*np.log10(pfb_win_norm_new/fft_window_norm)))
print()
print('Old predicted total  : %.4f dB' % (
      10*np.log10(pfb_win_norm_old/fft_window_norm) + 1.7606))
print('New predicted total  : %.4f dB' % (
      10*np.log10(pfb_win_norm_new/fft_window_norm) + 1.7606))
print('Target (ENBW only)   : +1.7606 dB')

fft_window_norm      : 6143.6250
pfb_win_norm (old)   : 12719.9868
pfb_win_norm (new)   : 16385.1971

Old norm offset      : 3.1606 dB
New norm offset      : 4.2603 dB

Old predicted total  : 4.9212 dB
New predicted total  : 6.0209 dB
Target (ENBW only)   : +1.7606 dB


In [10]:
import numpy as np

N_FFT_COARSE = 16384
N_TAPS       = 4
pfb_len      = N_FFT_COARSE * N_TAPS

t_pfb      = np.arange(pfb_len, dtype=np.float64) - pfb_len // 2
pfb_coeffs = np.sinc(t_pfb / N_FFT_COARSE) * np.hanning(pfb_len)
pfb_coeffs /= np.sum(pfb_coeffs.reshape(N_TAPS, N_FFT_COARSE), axis=0).mean()

fft_window      = np.hanning(N_FFT_COARSE).astype(np.float64)
fft_window_norm = float(np.sum(fft_window**2))

# The fix: scale the pfb norm to match fft_window_norm
# Physical reasoning: the PFB signal normalisation step already ensures
# unit DC gain per output bin. The power norm should therefore reference
# the same N-point window energy as the FFT, not the K*N prototype energy.
# The correct norm is fft_window_norm scaled by the square of the
# signal normalisation factor (since pfb_coeffs were divided by scale_factor).

scale_factor     = np.sum(
    (np.sinc(np.arange(pfb_len, dtype=np.float64) - pfb_len//2) / N_FFT_COARSE *
     np.hanning(pfb_len)).reshape(N_TAPS, N_FFT_COARSE), axis=0).mean()

# The Hann window norm scales as sum(w**2).
# After dividing pfb_coeffs by scale_factor, the effective per-bin
# energy is fft_window_norm / scale_factor**2 per tap.
# With K taps summed: K * fft_window_norm / scale_factor**2
# But the signal norm already accounts for scale_factor in the coefficients.
# The simplest and most consistent fix is:
pfb_win_norm_fixed = fft_window_norm

print('fft_window_norm        : %.6f' % fft_window_norm)
print('pfb_win_norm_fixed     : %.6f' % pfb_win_norm_fixed)
print('Norm offset            : 0.0000 dB (by construction)')
print('Predicted total offset : +1.7606 dB (ENBW only)')
print()
print('Verification with Gaussian noise:')

# Quick Gaussian test with the fixed norm
rng     = np.random.default_rng(42)
N_AVG   = 500
sigma   = 100.0
win     = np.hanning(N_FFT_COARSE).astype(np.float64)
win_norm = float(np.sum(win**2))

fft_levels = []
pfb_levels = []

for _ in range(N_AVG):
    noise = rng.normal(0, sigma, N_FFT_COARSE + pfb_len).astype(np.float64)
    # FFT
    fft_out = np.abs(np.fft.rfft(noise[-N_FFT_COARSE:] * win))**2 / win_norm
    fft_levels.append(float(np.mean(10*np.log10(fft_out[1:-1] + 1e-100))))
    # PFB with fixed norm
    blocks  = noise[:pfb_len].reshape(N_TAPS, N_FFT_COARSE)
    pfb_f   = np.sum(blocks * pfb_coeffs.reshape(N_TAPS, N_FFT_COARSE), axis=0)
    pfb_out = np.abs(np.fft.rfft(pfb_f))**2 / pfb_win_norm_fixed
    pfb_levels.append(float(np.mean(10*np.log10(pfb_out[1:-1] + 1e-100))))

fft_mean = np.mean(fft_levels)
pfb_mean = np.mean(pfb_levels)
measured = pfb_mean - fft_mean

print('FFT mean level         : %.4f dB' % fft_mean)
print('PFB mean level         : %.4f dB' % pfb_mean)
print('Measured PFB-FFT offset: %+.4f dB' % measured)
print('Target                 : +1.76 dB')
print('Difference from target : %+.4f dB' % (measured - 1.7606))
if abs(measured - 1.7606) < 0.1:
    print('VERDICT: PASS — fix is correct')
else:
    print('VERDICT: FAIL — further investigation needed')

fft_window_norm        : 6143.625000
pfb_win_norm_fixed     : 6143.625000
Norm offset            : 0.0000 dB (by construction)
Predicted total offset : +1.7606 dB (ENBW only)

Verification with Gaussian noise:
FFT mean level         : 37.4937 dB
PFB mean level         : 40.6523 dB
Measured PFB-FFT offset: +3.1586 dB
Target                 : +1.76 dB
Difference from target : +1.3980 dB
VERDICT: FAIL — further investigation needed


In [11]:
import numpy as np

N_FFT_COARSE = 16384
N_TAPS       = 4
pfb_len      = N_FFT_COARSE * N_TAPS
sigma        = 100.0
N_AVG        = 1000

# Raw prototype filter — no signal normalisation
t_pfb     = np.arange(pfb_len, dtype=np.float64) - pfb_len // 2
pfb_raw   = np.sinc(t_pfb / N_FFT_COARSE) * np.hanning(pfb_len)

# FFT window
fft_window     = np.hanning(N_FFT_COARSE).astype(np.float64)
fft_win_norm   = float(np.sum(fft_window**2))

# PFB norm — sum of squared raw coefficients
pfb_win_norm_raw = float(np.sum(pfb_raw**2))

print('fft_win_norm         : %.4f' % fft_win_norm)
print('pfb_win_norm_raw     : %.4f' % pfb_win_norm_raw)
print('Ratio pfb/fft        : %.4f' % (pfb_win_norm_raw / fft_win_norm))
print('Norm offset          : %.4f dB' % (10*np.log10(pfb_win_norm_raw / fft_win_norm)))
print()

# Gaussian noise test
rng = np.random.default_rng(42)
fft_levels = []
pfb_levels = []

pfb_reshaped = pfb_raw.reshape(N_TAPS, N_FFT_COARSE)

for _ in range(N_AVG):
    noise = rng.normal(0, sigma, N_FFT_COARSE + pfb_len).astype(np.float64)

    # FFT
    fft_out = np.abs(np.fft.rfft(
        noise[-N_FFT_COARSE:] * fft_window))**2 / fft_win_norm
    fft_levels.append(float(np.mean(fft_out[1:-1])))

    # PFB with raw coefficients
    blocks  = noise[:pfb_len].reshape(N_TAPS, N_FFT_COARSE)
    pfb_f   = np.sum(blocks * pfb_reshaped, axis=0)
    pfb_out = np.abs(np.fft.rfft(pfb_f))**2 / pfb_win_norm_raw
    pfb_levels.append(float(np.mean(pfb_out[1:-1])))

fft_mean_lin = np.mean(fft_levels)
pfb_mean_lin = np.mean(pfb_levels)

print('=== Linear power (should both equal sigma^2 = %.1f) ===' % sigma**2)
print('FFT mean power  : %.2f  (sigma^2 = %.1f)' % (fft_mean_lin, sigma**2))
print('PFB mean power  : %.2f  (sigma^2 = %.1f)' % (pfb_mean_lin, sigma**2))
print('PFB/FFT ratio   : %.6f' % (pfb_mean_lin / fft_mean_lin))
print('Offset          : %+.4f dB' % (10*np.log10(pfb_mean_lin / fft_mean_lin)))
print()
print('=== In dB ===')
fft_db = 10*np.log10(fft_mean_lin)
pfb_db = 10*np.log10(pfb_mean_lin)
print('FFT mean        : %.4f dB' % fft_db)
print('PFB mean        : %.4f dB' % pfb_db)
print('PFB - FFT       : %+.4f dB' % (pfb_db - fft_db))
print()
if abs(pfb_mean_lin / fft_mean_lin - 1.0) < 0.01:
    print('VERDICT: PASS — both spectrometers give sigma^2 per bin')
    print('Fix: use pfb_win_norm = sum(pfb_raw**2) WITHOUT signal normalisation')
else:
    print('VERDICT: FAIL — offset remains')
    print('Residual: %.4f dB' % (10*np.log10(pfb_mean_lin/fft_mean_lin)))

fft_win_norm         : 6143.6250
pfb_win_norm_raw     : 13045.2090
Ratio pfb/fft        : 2.1234
Norm offset          : 3.2703 dB

=== Linear power (should both equal sigma^2 = 10000.0) ===
FFT mean power  : 9998.53  (sigma^2 = 10000.0)
PFB mean power  : 10001.47  (sigma^2 = 10000.0)
PFB/FFT ratio   : 1.000294
Offset          : +0.0013 dB

=== In dB ===
FFT mean        : 39.9994 dB
PFB mean        : 40.0006 dB
PFB - FFT       : +0.0013 dB

VERDICT: PASS — both spectrometers give sigma^2 per bin
Fix: use pfb_win_norm = sum(pfb_raw**2) WITHOUT signal normalisation


In [1]:
import os, glob
import numpy as np

BASE = '/Users/user/Downloads/Manny-Masters/Project/Data'

for root, dirs, files in os.walk(BASE):
    npy_files = [f for f in files if f.endswith('.npy') or f.endswith('.png')]
    if npy_files:
        print('\n' + '='*60)
        print(root.replace(BASE, ''))
        print('='*60)
        for f in sorted(npy_files):
            full = os.path.join(root, f)
            if f.endswith('.npy'):
                try:
                    arr = np.load(full, allow_pickle=False)
                    print('  %-55s shape=%s' % (f, str(arr.shape)))
                except:
                    print('  %-55s (unreadable)' % f)
            else:
                size_kb = os.path.getsize(full) / 1024
                print('  %-55s %.0f KB' % (f, size_kb))


/Jodrell_load/LNA
  fft_coarse_20250619_211156.npy                          shape=(8193,)
  fft_hires_20250619_211156.npy                           shape=(524289,)
  freq_coarse_20250619_211156.npy                         shape=(8193,)
  freq_hires_20250619_211156.npy                          shape=(524289,)
  integ_final_20250619_204116.npy                         shape=(524289,)
  integ_freq_20250619_204116.npy                          shape=(524289,)
  integ_snap_20250619_204116_N105.npy                     shape=(524289,)
  integ_snap_20250619_204116_N116.npy                     shape=(524289,)
  integ_snap_20250619_204116_N12.npy                      shape=(524289,)
  integ_snap_20250619_204116_N128.npy                     shape=(524289,)
  integ_snap_20250619_204116_N139.npy                     shape=(524289,)
  integ_snap_20250619_204116_N151.npy                     shape=(524289,)
  integ_snap_20250619_204116_N162.npy                     shape=(524289,)
  integ_snap_20250619_2